# STAIR-Enhanced v2a — Residual-Whitening Projector
## Kaggle Training Notebook (Micro-Ablation — ResOnly)

| | |
|---|---|
| **Model** | EnhancedSTAIR_v2a (whitening + λ·Δ residual) |
| **Datasets** | Baby · Sports · Electronics |
| **Optimizer** | AdamWSEvo (embed) + Adam (projector, lr=5e-3) |
| **AMP** | ✅ `autocast` + `GradScaler` |
| **Baseline** | STAIR (main.py) — SVD Whitening only |
| **v1 Result** | −6.4% → −9.6% (structural prior mismatch) |
| **v2 Fix** | Keep whitening; add nonlinear residual correction |

> **Run order:** Cell 1→2→3→4 rồi chọn dataset cần chạy (Cell 5 hoặc Cell 6).


## Cell 1 — Setup & Install


In [ ]:
# ==========================================================
# CELL 1: SETUP — Môi trường Kaggle (T4 x2 / P100)
# ==========================================================
import subprocess, sys, os

# [FIX v2] Cài đặt đúng phiên bản — tránh lỗi torch==2.0.1 của torchdata
# torchdata==0.7.1 tương thích torch 2.2.x (Kaggle default)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec==0.9.7',
    'torchdata==0.7.1', '--no-deps',
    'nvidia-ml-py', 'prettytable',
], check=True)

# Clone STAIR-Enhanced repo
if not os.path.exists('/kaggle/working/STAIR-Enhanced'):
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/ThanhChuong12/STAIR-Enhanced.git',
        '/kaggle/working/STAIR-Enhanced',
    ], check=True)

os.chdir('/kaggle/working/STAIR-Enhanced')
print('CWD:', os.getcwd())

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Verify files cần thiết tồn tại
required = [
    'main_enhanced_v2.py',
    'models/residual_projector_v2.py',
    'optimizers/AdamW.py',
    'optimizers/utils.py',
]
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status}  {f}')


## Cell 2 — Copy Datasets từ Kaggle Input


In [ ]:
# ==========================================================
# CELL 2: COPY DATASETS
# Upload datasets qua Kaggle Dataset (input path bên dưới)
# ==========================================================
import shutil, os

KAGGLE_INPUT = '/kaggle/input'
DATA_ROOT    = '/kaggle/working/STAIR-Enhanced/data'
os.makedirs(DATA_ROOT, exist_ok=True)

DATASETS = {
    'Amazon2014Baby_550_MMRec':        'amazon2014baby550mmrec',
    'Amazon2014Sports_550_MMRec':      'amazon2014sports550mmrec',
    'Amazon2014Electronics_550_MMRec': 'amazon2014electronics550mmrec',
}

for ds_name, kaggle_slug in DATASETS.items():
    src = os.path.join(KAGGLE_INPUT, kaggle_slug, ds_name)
    dst = os.path.join(DATA_ROOT, ds_name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'  Copied: {ds_name}')
    elif os.path.exists(dst):
        print(f'  OK (exists): {ds_name}')
    else:
        print(f'  ⚠ NOT FOUND: {src}  (kiểm tra Dataset input slug)')


## Cell 3 — Verify Model Architecture & Warm-start


In [ ]:
# ==========================================================
# CELL 3: VERIFY — Architecture sanity check
# ==========================================================
import sys, os
sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')

import math, torch
import torch.nn as nn
from models.residual_projector_v2 import (
    ResidualWhiteningProjector, composite_embeddings
)

D_T, D_V, D_H = 384, 4096, 64
N_DEMO        = 500   # Dùng N nhỏ để test nhanh

proj = ResidualWhiteningProjector(d_text=D_T, d_visual=D_V, d_hidden=D_H, lambda_init=0.1)

# Dummy data
text_feat   = torch.randn(N_DEMO, D_T)
visual_feat = torch.randn(N_DEMO, D_V)
e_svd       = torch.randn(N_DEMO, D_H).mul(math.sqrt(N_DEMO / D_H))

# Warm-start
proj.init_warm_start_weights(text_feat, visual_feat)

# Forward check
with torch.no_grad():
    delta   = proj(text_feat, visual_feat)
    e_final = composite_embeddings(proj, text_feat, visual_feat, e_svd, N_DEMO, D_H)

print(f'Delta shape    : {tuple(delta.shape)}')
print(f'Delta L2-norm  : {delta.norm(dim=-1).mean():.6f}  (expected ~1.0)')
print(f'e_final shape  : {tuple(e_final.shape)}')
print(f'lambda_res     : {proj.lambda_res.item():.4f}  requires_grad={proj.lambda_res.requires_grad}')

# Residual contribution magnitude
correction_magnitude = (proj.lambda_res.item() * delta).norm(dim=-1).mean().item()
svd_magnitude        = e_svd.norm(dim=-1).mean().item()
print(f'||e_svd||      : {svd_magnitude:.3f}')
print(f'||λ·Δ||        : {correction_magnitude:.3f}  ({100*correction_magnitude/svd_magnitude:.1f}% của e_svd)')
print()
print('✅ Architecture v2a (ResOnly) sẵn sàng!')
print('  e_i = whitening(x_i)  +  λ_res · Δ_i')
print(f'  Tổng tham số projector: {sum(p.numel() for p in proj.parameters()):,}')


## Cell 4 — Training Helpers (Log Parser + Profiler)


In [ ]:
# ==========================================================
# CELL 4: TRAINING HELPERS
# ==========================================================
import re, json, time
from pathlib import Path
import torch

# ---- VRAM & Timing Profiler ----
class TrainingProfiler:
    """Theo dõi VRAM và thời gian training."""
    def __init__(self):
        self.start_time = time.time()
        self.epoch_times = []
    def snapshot(self, epoch):
        elapsed = time.time() - self.start_time
        vram = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0
        self.epoch_times.append({'epoch': epoch, 'elapsed': elapsed, 'vram_gb': vram})
        return vram
    def summary(self):
        if not self.epoch_times: return
        total = time.time() - self.start_time
        max_vram = max(e['vram_gb'] for e in self.epoch_times)
        print(f'Total time : {total/60:.1f} min')
        print(f'Max VRAM   : {max_vram:.2f} GB')

# ---- Log Parser ----
def parse_log(log_path: str) -> dict:
    """Parse freerec log để lấy best TEST metrics và lambda_res evolution."""
    if not Path(log_path).exists():
        return {}
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        content = f.read()
    flat = content.replace('\n', ' ')

    # Best TEST metrics
    m = re.search(
        r'Load best model @Epoch[: ]+([0-9]+).*?TEST.*?'
        r'RECALL@10 Avg: ([0-9.]+).*?RECALL@20 Avg: ([0-9.]+).*?'
        r'NDCG@10 Avg: ([0-9.]+).*?NDCG@20 Avg: ([0-9.]+)',
        flat
    )
    metrics = {}
    if m:
        metrics = {
            'best_epoch': int(m.group(1)),
            'R@10':  float(m.group(2)),
            'R@20':  float(m.group(3)),
            'N@10':  float(m.group(4)),
            'N@20':  float(m.group(5)),
        }

    # lambda_res evolution (dòng log từ CoachForEnhancedSTAIR_v2a)
    lam_entries = re.findall(
        r'\[lambda_res @epoch\s*(\d+)\].*?mean=([0-9.]+)', content
    )
    metrics['lambda_res'] = [(int(e), float(v)) for e, v in lam_entries]

    return metrics

# ---- run_training() ----
def run_training(dataset_name: str, log_path: str, extra_args: list = None):
    """
    Chạy main_enhanced_v2.py với config YAML và capture log.
    extra_args: list các cờ thêm, ví dụ ['--lr-proj', '0.005']
    """
    import subprocess, sys
    cfg_file = f'configs/{dataset_name}.yaml'
    cmd = [
        sys.executable, 'main_enhanced_v2.py',
        '--config', cfg_file,
    ]
    if extra_args:
        cmd.extend(extra_args)
    print(f'[run] {" ".join(cmd)}')
    print(f'[run] Log -> {log_path}')
    print('-' * 60)
    with open(log_path, 'w', encoding='utf-8') as log_f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
            cwd='/kaggle/working/STAIR-Enhanced',
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
            log_f.write(line)
    rc = proc.wait()
    print(f'\n[run] Return code: {rc}')
    return rc

print('✅ Helpers loaded: TrainingProfiler, parse_log, run_training')


## Cell 5 — [COMPLETED] Train Baby + Sports


In [ ]:
# [COMPLETED] Cell nay da chay xong. Bo comment de chay lai.
# # ==========================================================
# # CELL 5: TRAIN BABY + SPORTS (v2a)
# # [COMPLETED] Bỏ comment để chạy lại
# # ==========================================================
# import os; os.chdir('/kaggle/working/STAIR-Enhanced')
# 
# # Baby
# rc_baby = run_training(
#     dataset_name='Amazon2014Baby_550_MMRec',
#     log_path='/kaggle/working/log_v2a_baby.txt',
# )
# 
# # Sports
# rc_sports = run_training(
#     dataset_name='Amazon2014Sports_550_MMRec',
#     log_path='/kaggle/working/log_v2a_sports.txt',
# )
# 
# print(f'Baby   rc={rc_baby}')
# print(f'Sports rc={rc_sports}')


## Cell 6 — [ACTIVE] Train Electronics


In [ ]:
# ==========================================================
# CELL 6: TRAIN ELECTRONICS (v2a)
# Tách riêng vì Electronics cần nhiều RAM/VRAM hơn
# batch_size=4096 được set sẵn trong YAML
# ==========================================================
import os; os.chdir('/kaggle/working/STAIR-Enhanced')

rc_elec = run_training(
    dataset_name='Amazon2014Electronics_550_MMRec',
    log_path='/kaggle/working/log_v2a_electronics.txt',
)
print(f'Electronics rc={rc_elec}')


## Cell 7 — So sánh v2a vs Baseline vs v1


In [ ]:
# ==========================================================
# CELL 7: PARSE LOGS & SO SÁNH
# ==========================================================
from prettytable import PrettyTable

# Kết quả Baseline (từ tái lập thực nghiệm — logs/paper/)
BASELINE = {
    'Baby':        {'R@10': 0.0674, 'R@20': 0.1042, 'N@10': 0.0359, 'N@20': 0.0454},
    'Sports':      {'R@10': 0.0743, 'R@20': 0.1111, 'N@10': 0.0405, 'N@20': 0.0500},
    'Electronics': {'R@10': 0.0442, 'R@20': 0.0665, 'N@10': 0.0246, 'N@20': 0.0303},
}

# Kết quả v1 (Enhanced v1 — DeRedundantGatedProjector, từ thực nghiệm trước)
V1_RESULTS = {
    'Baby':        {'R@10': 0.0611, 'R@20': 0.0948, 'N@10': 0.0325, 'N@20': 0.0412},
    'Sports':      {'R@10': 0.0695, 'R@20': 0.1040, 'N@10': 0.0376, 'N@20': 0.0466},
    'Electronics': {'R@10': 0.0401, 'R@20': 0.0601, 'N@10': 0.0223, 'N@20': 0.0274},
}

# Parse kết quả v2a từ log
LOG_MAP = {
    'Baby':        '/kaggle/working/log_v2a_baby.txt',
    'Sports':      '/kaggle/working/log_v2a_sports.txt',
    'Electronics': '/kaggle/working/log_v2a_electronics.txt',
}

v2a_results = {}
for ds, log_path in LOG_MAP.items():
    parsed = parse_log(log_path)
    v2a_results[ds] = parsed if parsed else {}
    print(f'{ds}: best_epoch={parsed.get("best_epoch","N/A")}')

# ---- Bảng so sánh tổng hợp ----
def pct(v2, base):
    if v2 and base: return f'{(v2/base-1)*100:+.1f}%'
    return 'N/A'

table = PrettyTable()
table.field_names = ['Dataset', 'Model', 'R@10', 'R@20', 'N@10', 'N@20',
                     'ΔR@10', 'ΔN@20']
table.align = 'r'; table.align['Dataset'] = 'l'; table.align['Model'] = 'l'

for ds in ['Baby', 'Sports', 'Electronics']:
    b = BASELINE[ds]
    v1 = V1_RESULTS[ds]
    v2 = v2a_results.get(ds, {})
    # Baseline
    table.add_row([ds, 'Baseline',
        f'{b["R@10"]:.4f}', f'{b["R@20"]:.4f}',
        f'{b["N@10"]:.4f}', f'{b["N@20"]:.4f}',
        '—', '—'])
    # v1
    table.add_row([ds, 'Enhanced-v1 (Replace)',
        f'{v1["R@10"]:.4f}', f'{v1["R@20"]:.4f}',
        f'{v1["N@10"]:.4f}', f'{v1["N@20"]:.4f}',
        pct(v1['R@10'], b['R@10']), pct(v1['N@20'], b['N@20'])])
    # v2a
    if v2 and 'R@10' in v2:
        table.add_row([ds, 'Enhanced-v2a (Residual)',
            f'{v2["R@10"]:.4f}', f'{v2["R@20"]:.4f}',
            f'{v2["N@10"]:.4f}', f'{v2["N@20"]:.4f}',
            pct(v2['R@10'], b['R@10']), pct(v2['N@20'], b['N@20'])])
    else:
        table.add_row([ds, 'Enhanced-v2a (Residual)',
            '—', '—', '—', '—', '—', '—'])
    table.add_row(['-'*12, '-'*22, '-'*6, '-'*6, '-'*6, '-'*6, '-'*7, '-'*7])

print(table)


## Cell 8 — lambda_res Evolution & Learning Curves


In [ ]:
# ==========================================================
# CELL 8: PLOT lambda_res & Learning Curves
# ==========================================================
import matplotlib.pyplot as plt
import re

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
datasets = ['Baby', 'Sports', 'Electronics']
log_paths = [
    '/kaggle/working/log_v2a_baby.txt',
    '/kaggle/working/log_v2a_sports.txt',
    '/kaggle/working/log_v2a_electronics.txt',
]

for ax, ds, lp in zip(axes, datasets, log_paths):
    parsed = parse_log(lp)
    lam_data = parsed.get('lambda_res', [])
    if lam_data:
        epochs  = [x[0] for x in lam_data]
        lambdas = [x[1] for x in lam_data]
        ax.plot(epochs, lambdas, 'b-', linewidth=2, label='lambda_res')
        ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='init=0.1')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('λ_res')
        ax.set_title(f'{ds} — λ_res Evolution')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No data yet', ha='center', va='center',
                transform=ax.transAxes, fontsize=14, color='gray')
        ax.set_title(f'{ds} — λ_res Evolution (No data)')

plt.suptitle('STAIR-Enhanced v2a: Residual Scaling Parameter (λ_res) Evolution',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/lambda_res_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/lambda_res_evolution.png')

# ---- Interpretation guide ----
print()
print('=== Interpretation Guide ===')
print('λ_res > 0.1 → Residual correction mang lại lợi ích, optimizer tăng λ')
print('λ_res < 0.1 → Correction gây hại, optimizer giảm λ (tốt!)')
print('λ_res ≈ 0.1 → Correction trung tính, embedding gần như baseline')
print('λ_res ↗ monotone → Projector đang học features bổ sung tốt')
print('λ_res oscillates → lr_proj có thể quá cao, thử giảm xuống 1e-3')


## Cell 9 — Export Kết quả CSV


In [ ]:
# ==========================================================
# CELL 9: EXPORT CSV
# ==========================================================
import csv, os
from pathlib import Path

OUT_CSV = '/kaggle/working/stair_v2a_results.csv'

rows = []
for ds in ['Baby', 'Sports', 'Electronics']:
    b = BASELINE[ds]
    v1 = V1_RESULTS[ds]
    v2 = v2a_results.get(ds, {})
    for model, m in [('Baseline', b), ('Enhanced-v1', v1), ('Enhanced-v2a', v2)]:
        if not m or 'R@10' not in m: continue
        rows.append({
            'dataset': ds, 'model': model,
            'R@10': m['R@10'], 'R@20': m['R@20'],
            'N@10': m['N@10'], 'N@20': m['N@20'],
            'best_epoch': m.get('best_epoch', ''),
        })

with open(OUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset','model','R@10','R@20','N@10','N@20','best_epoch'])
    writer.writeheader()
    writer.writerows(rows)

print(f'✅ Exported {len(rows)} rows -> {OUT_CSV}')
print(Path(OUT_CSV).read_text()[:500])


## Cell 10 — Checklist & Kết luận

### ✅ Checklist sau khi chạy xong

- [ ] Cell 5: Baby + Sports finished (rc=0)
- [ ] Cell 6: Electronics finished (rc=0)
- [ ] Cell 7: Bảng so sánh đã hiển thị đầy đủ 3 dataset × 3 model
- [ ] Cell 8: Plot lambda_res evolution đã lưu PNG
- [ ] Cell 9: CSV đã export

### Tiêu chí Thành công (từ STAIRE2_v2_Report.md)

| Ngưỡng | Điều kiện |
|---|---|
| **Tối thiểu** | v2a ≥ Baseline trên mọi dataset (không regression) |
| **Kỳ vọng** | +2% trở lên trên ≥1 dataset |
| **Lý tưởng** | +3%→+5% trên Baby |

### Nếu v2a vẫn tệ hơn Baseline

Các bước debug theo thứ tự:
1. Kiểm tra plot lambda_res: nếu λ ↘ về 0 → projector học được rằng correction không có ích → **giảm lambda_init=0.01**
2. Nếu λ tăng nhanh vượt 0.5 → projector dominant, mất structural prior → **giảm lr_proj=1e-3**
3. Thử `--lambda-init 0.0` (thuần baseline) để xác nhận framing đúng
4. Chuyển sang v2b (GateOnly) như kế hoạch Micro-Ablation
